# Домашнє завдання до модуля «Текст як послідовність»

In [1]:
# Завантажити англо-польський набір Europarl і зберегти його в пам'яті.

from datasets import load_dataset

dataset = load_dataset("Helsinki-NLP/europarl", "en-pl")
dataset

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 631160
    })
})

In [2]:
# Перевірити доступні розділи/ключі набору даних.

dataset.keys()

dict_keys(['train'])

In [3]:
# Вибрати тренувальний розділ і переглянути перший приклад.

split_data = dataset['train']
split_data[0]

{'translation': {'en': "Action taken on Parliament's resolutions: see Minutes",
  'pl': 'Działania podjęte w wyniku rezolucji Parlamentu: Patrz protokól'}}

In [4]:
# Завантажити мовні моделі spaCy для токенізації англійською та польською.

!python -m spacy download en_core_web_sm
!python -m spacy download pl_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 22.1 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 35.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.3.1 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('pl_core_news_sm')


In [5]:
import os
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

In [6]:
# Описати токенізатор, що будує словники і числово кодує набір даних.

import spacy
from collections import Counter
from torchtext.vocab import Vocab
from typing import List

class Tokenizer:
    def __init__(
        self, 
        dataset, 
        langs: List[str], 
        space_models: List[str],
        unk_token: str = "<unk>",
        pad_token: str = "<pad>",
        bos_token: str = "<bos>",
        eos_token: str = "<eos>",
        max_samples: int = None
    ):
        self.dataset = dataset
        self.langs = langs
        self.space_models = space_models
        self.unk_token = unk_token
        self.pad_token = pad_token
        self.bos_token = bos_token
        self.eos_token = eos_token
        self.max_samples = max_samples
        self.vocab = None
        self.dataset.langs = langs

        # Завантаження spacy моделей в ініціалізаторі
        self.nlp_models = {}
        for lang, model_name in zip(self.langs, self.space_models):
            nlp = spacy.load(model_name)
            nlp.disable_pipes(*[pipe for pipe in nlp.pipe_names if pipe not in ["tokenizer"]])
            self.nlp_models[lang] = nlp

    @property
    def pad_index(self):
        """Отримати індекс pad токена як property"""
        if self.vocab is None:
            raise ValueError("Vocabulary not created yet. Call tokenizer first.")

        return self.vocab[f"{self.langs[0]}_vocab"][self.pad_token]

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        pass

    def create_vocabulary(self):
        special_tokens = [self.unk_token, self.pad_token, self.bos_token, self.eos_token]
        vocab = {}

        dataset_to_process = self.dataset.select(range(min(self.max_samples, len(self.dataset)))) if self.max_samples else self.dataset

        for lang in self.langs:
            counter = Counter()
            nlp = self.nlp_models[lang]

            texts = [example["translation"][lang] for example in dataset_to_process]

            for doc in nlp.pipe(texts, batch_size=1000, n_process=1):
                tokens = [token.text.lower() for token in doc]
                counter.update(tokens)

            filtered_counter = Counter({token: count for token, count in counter.items() if count >= 2})

            # Використовуємо build_vocab_from_iterator замість прямого Vocab
            from torchtext.vocab import build_vocab_from_iterator
            
            # Створюємо ітератор з відфільтрованих токенів
            token_iterator = [[token] for token in filtered_counter.keys()]
            
            vocab_obj = build_vocab_from_iterator(
                token_iterator,
                specials=special_tokens,
                special_first=True
            )
            
            # Встановлюємо default_index для невідомих токенів
            unk_idx = vocab_obj[self.unk_token]
            vocab_obj.set_default_index(unk_idx)
            
            vocab[f"{lang}_vocab"] = vocab_obj

        return vocab

    def tokenize_example(self, examples):
        """Tokenize a batch of examples"""
        result = {}

        for lang, nlp in self.nlp_models.items():
            texts = [translation[lang] for translation in examples["translation"]]

            tokenized_texts = []
            for text in texts:
                doc = nlp(text)
                tokens = [token.text.lower() for token in doc if not token.is_punct and not token.is_space]
                tokenized_texts.append(tokens)

            result[f"{lang}_tokens"] = tokenized_texts

        return result

    def numericalize_example(self, example, vocab):
        """Convert tokenized text to numerical indices using vocabulary."""
        for lang in self.langs:
            vocab_key = f"{lang}_vocab"
            tokens_key = f"{lang}_tokens"
            ids_key = f"{lang}_ids"

            vocab_obj = vocab[vocab_key]

            if isinstance(example[tokens_key], list) and len(example[tokens_key]) > 0:
                if isinstance(example[tokens_key][0], list):
                    ids = [[vocab_obj[token] for token in tokens]
                           for tokens in example[tokens_key]]
                else:
                    ids = [vocab_obj[token] for token in example[tokens_key]]
            else:
                ids = []

            example[ids_key] = ids

        return example

    def __call__(self, *args, **kwargs) -> Vocab:
        self.vocab = self.create_vocabulary()

        return self.dataset.map(
            self.tokenize_example,
            batched=True,
            num_proc=1
        ).map(
            lambda x: self.numericalize_example(x, self.vocab),
            batched=True,
            num_proc=1
        )

In [7]:
# Описати обгортку DataLoader з батчингом і паддінгом.

from torch import nn
import torch


class DataLoaderGetter:
    def __init__(self, tokenized_dataset, pad_index, langs, batch_size=32, shuffle=False):
        """
        Приймає готові токенізовані дані та параметри

        Args:
            tokenized_dataset: Токенізований dataset
            pad_index: Індекс padding токена
            langs: Список мов
            batch_size: Розмір батча
            shuffle: Перемішувати дані
        """
        self.dataset = tokenized_dataset
        self.pad_index = pad_index
        self.langs = langs
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __enter__(self):
        return self()

    def __exit__(self, exc_type, exc_val, exc_tb):
        pass

    def collate_fn(self, batch):
        return {
            f"{lang}_ids": nn.utils.rnn.pad_sequence(
                [torch.tensor(example[f"{lang}_ids"]) for example in batch],
                padding_value=self.pad_index
            )
            for lang in self.langs
        }

    def __call__(self):
        return torch.utils.data.DataLoader(
            dataset=self.dataset,
            batch_size=self.batch_size,
            shuffle=self.shuffle,
            collate_fn=self.collate_fn
        )

In [ ]:
# Описати енкодер GRU для моделі seq2seq.

class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, encoder_hidden_dim, decoder_hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.GRU(embedding_dim, encoder_hidden_dim, bidirectional = True)
        self.fc = nn.Linear(encoder_hidden_dim * 2, decoder_hidden_dim)

    def forward(self, src): # (src_length, batch size)
        embedded = self.embedding(src) # (src_length, batch_size, embedding_dim)

        outputs, hidden = self.rnn(embedded)
        # outputs (src_length, batch_size, hidden dim * n_directions)
        # hidden (n_layers * n_directions, batch_size, hidden dim)

        # hidden is stacked [forward_1, backward_1, forward_2, backward_2, ...]

        # outputs are always from the last layer
        # hidden [-2, :, : ] is the last of the forwards RNN
        # hidden [-1, :, : ] is the last of the backwards RNN

        # initial decoder hidden is final hidden state of the forwards and backwards
        # encoder RNNs fed through a linear layer

        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))

        # outputs (src_length, batch_size, encoder_hidden_dim * 2)
        # hidden (batch_size, decoder_hidden_dim)

        return outputs, hidden


In [ ]:
# Описати механізм уваги, який використовує декодер.

class Attention(nn.Module):
    def __init__(self, encoder_hidden_dim, decoder_hidden_dim):
        super().__init__()
        self.attn_fc = nn.Linear(
            (encoder_hidden_dim * 2) + decoder_hidden_dim,
            decoder_hidden_dim
        )
        self.v_fc = nn.Linear(decoder_hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden (batch_size, decoder_hidden_dim)
        # encoder_outputs (src_length, batch_size, encoder_hidden dim * 2)

        batch_size = encoder_outputs.shape[1]
        src_length = encoder_outputs.shape[0]

        # repeat decoder hidden state src_length times
        hidden = hidden.unsqueeze(1).repeat(1, src_length, 1) # (batch_size, src_length, decoder_hidden_dim)
        encoder_outputs = encoder_outputs.permute(1, 0, 2) # (batch_size, src_length, encoder_hidden_dim * 2)

        energy = torch.tanh(self.attn_fc(torch.cat((hidden, encoder_outputs), dim=2)))
        # (batch_size, src_length, decoder_hidden_dim)

        attention = self.v_fc(energy).squeeze(2) # batch_size, src_length

        return torch.softmax(attention, dim=1)


In [ ]:
# Описати декодер GRU з увагою для генерації послідовностей.

class Decoder(nn.Module):
    def __init__(
        self,
        output_dim,
        embedding_dim,
        encoder_hidden_dim,
        decoder_hidden_dim,
        attention,
    ):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.GRU((encoder_hidden_dim * 2) + embedding_dim, decoder_hidden_dim)
        self.fc_out = nn.Linear(
            (encoder_hidden_dim * 2) + decoder_hidden_dim + embedding_dim,
            output_dim
        )

    def forward(self, input, hidden, encoder_outputs):
        # input = (batch_size)
        # hidden = batch_size, decoder hidden dim]
        # encoder_outputs = [src length, batch size, encoder hidden dim * 2]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.embedding(input)
        #embedded = [1, batch size, embedding dim]
        a = self.attention(hidden, encoder_outputs)
        # a = [batch size, src length]
        a = a.unsqueeze(1)
        # a = [batch size, 1, src length]
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        # encoder_outputs = [batch size, src length, encoder hidden dim * 2]
        weighted = torch.bmm(a, encoder_outputs) # batch matrix-matrix product
        # weighted = [batch size, 1, encoder hidden dim * 2]
        weighted = weighted.permute(1, 0, 2)
        # weighted = [1, batch size, encoder hidden dim * 2]
        rnn_input = torch.cat((embedded, weighted), dim = 2)
        # rnn_input = [1, batch size, (encoder hidden dim * 2) + embedding dim]
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        # output = [seq length, batch size, decoder hid dim * n directions]
        # hidden = [n layers * n directions, batch size, decoder hid dim]
        # seq len, n layers and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, decoder hidden dim]
        # hidden = [1, batch size, decoder hidden dim]
        # this also means that output == hidden
        assert (output == hidden).all()
        embedded = embedded.squeeze(0)
        output = output.squeeze(0)
        weighted = weighted.squeeze(0)
        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))
        #prediction = [batch size, output dim]
        return prediction, hidden.squeeze(0), a.squeeze(1)



In [ ]:
import random


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use teacher forcing 75% of the time
        batch_size = src.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # encoder_outputs is all hidden states of the input sequence, back and forwards
        # hidden is the final forward and backward hidden states, passed through a linear layer
        encoder_outputs, hidden = self.encoder(src)
        # outputs = [src length, batch size, encoder hidden dim * 2]
        # hidden = [batch size, decoder hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0,:]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden state and all encoder hidden states
            # receive output tensor (predictions) and new hidden state
            output, hidden, _ = self.decoder(input, hidden, encoder_outputs)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, decoder hidden dim]
            #place predictions in a tensor holding predictions for each token
            outputs[t] = output
            #decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            #get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs


In [ ]:
import tqdm
import os
from torch import optim
import platform

class Seq2SeqTrainer:
    def __init__(self, dataset, input_dim: int, output_dim: int, pad_index: int,
                 teacher_forcing_ratio: float, clip: float, n_epochs: int,
                 encoder_embedding_dim: int=256, decoder_embedding_dim: int=256,
                 encoder_hidden_dim: int=512, decoder_hidden_dim: int=512,
                 model_dir: str='./saved_models') -> None:
        self.n_epochs = n_epochs
        self.clip = clip
        self.teacher_forcing_ratio = teacher_forcing_ratio
        self.dataset = dataset
        self.model_dir = model_dir

        # Динамічно визначаємо ключі мов
        first_batch = next(iter(self.dataset))
        self.lang_ids_keys = [k for k in first_batch.keys() if k.endswith("_ids")]

        # Ініціалізація моделі
        attention = Attention(encoder_hidden_dim, decoder_hidden_dim)
        encoder = Encoder(input_dim, encoder_embedding_dim, encoder_hidden_dim, decoder_hidden_dim)
        decoder = Decoder(output_dim, decoder_embedding_dim, encoder_hidden_dim,
                         decoder_hidden_dim, attention)

        self.model = Seq2Seq(encoder, decoder, self.device).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters())
        self.criterion = nn.CrossEntropyLoss(ignore_index=pad_index)
        self.model.apply(self.init_weights)

    @property
    def device(self) -> torch.device:
        """Повертає пристрій з пріоритетом CUDA > MPS > CPU"""
        if torch.cuda.is_available():
            return torch.device("cuda")
        elif platform.system().lower() == "darwin" and torch.backends.mps.is_available():
            return torch.device("mps")
        else:
            return torch.device("cpu")

    def init_weights(self, m):
        """Ініціалізація ваг моделі"""
        for name, param in m.named_parameters():
            if "weight" in name:
                nn.init.normal_(param.data, mean=0, std=0.01)
            else:
                nn.init.constant_(param.data, 0)

    def train_fn(self):
        """Тренування з захистом від витоків пам'яті"""
        self.model.train()
        epoch_loss = 0
        batch_count = 0

        for batch in self.dataset:
            try:
                # Перенесення даних на пристрій
                src, trg = tuple(batch[k].to(self.device) for k in self.lang_ids_keys)

                self.optimizer.zero_grad()

                output = self.model(src, trg, self.teacher_forcing_ratio)
                output_reshaped = output[1:].reshape(-1, output.shape[-1])
                trg_reshaped = trg[1:].reshape(-1)

                loss = self.criterion(output_reshaped, trg_reshaped)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.clip)
                self.optimizer.step()

                epoch_loss += loss.item()
                batch_count += 1

                # Очищення кешу кожні 10 батчів для запобігання накопиченню пам'яті
                if batch_count % 10 == 0:
                    if self.device.type == "cuda":
                        torch.cuda.empty_cache()
                    elif self.device.type == "mps":
                        torch.mps.empty_cache()

                # Видалення тензорів
                del output, output_reshaped, trg_reshaped, loss, src, trg

            except RuntimeError as e:
                if "out of memory" in str(e) or "CUDA" in str(e):
                    print(f"⚠️ Memory error in batch {batch_count}: {e}")
                    # Очищення і пропуск батча
                    if self.device.type == "cuda":
                        torch.cuda.empty_cache()
                    elif self.device.type == "mps":
                        torch.mps.empty_cache()
                    continue
                else:
                    raise e

        return epoch_loss / max(batch_count, 1)

    def evaluate_fn(self):
        """Оцінка моделі з захистом від витоків пам'яті"""
        self.model.eval()
        epoch_loss = 0
        batch_count = 0

        with torch.no_grad():
            for batch in self.dataset:
                try:
                    src, trg = tuple(batch[k].to(self.device) for k in self.lang_ids_keys)

                    output = self.model(src, trg, 0)  # без teacher forcing
                    output_reshaped = output[1:].reshape(-1, output.shape[-1])
                    trg_reshaped = trg[1:].reshape(-1)

                    loss = self.criterion(output_reshaped, trg_reshaped)
                    epoch_loss += loss.item()
                    batch_count += 1

                    # Очищення
                    del output, output_reshaped, trg_reshaped, loss, src, trg

                except RuntimeError as e:
                    if "out of memory" in str(e) or "CUDA" in str(e):
                        print(f"⚠️ Memory error in evaluation batch {batch_count}")
                        if self.device.type == "cuda":
                            torch.cuda.empty_cache()
                        elif self.device.type == "mps":
                            torch.mps.empty_cache()
                        continue
                    else:
                        raise e

        return epoch_loss / max(batch_count, 1)

    def train(self):
        """Основний цикл тренування"""
        first_batch = next(iter(self.dataset))
        lang_keys = [k.replace("_ids", "") for k in self.lang_ids_keys]

        best_valid_loss = float('inf')

        for epoch in tqdm.tqdm(range(self.n_epochs), desc="Training"):
            train_loss = self.train_fn()
            valid_loss = self.evaluate_fn()

            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                os.makedirs(self.model_dir, exist_ok=True)
                torch.save(self.model.state_dict(),
                          os.path.join(self.model_dir, f'{"_".join(lang_keys)}.pt'))

            print(f"\tEpoch {epoch+1}/{self.n_epochs} - Train Loss: {train_loss:.3f} | Valid Loss: {valid_loss:.3f}")

        return best_valid_loss

In [ ]:
import torch
import optuna
import gc

class HyperParametersOptimizer:
    def __init__(self, dataset, n_trials: int, batch_size=32):
        self.dataset = dataset
        self.n_trials = n_trials
        self.batch_size = batch_size

        # Динамічно витягуємо мови
        first_example = self.dataset.dataset[0]
        self.lang_keys = list(first_example['translation'].keys())

        # Безпечне визначення vocab sizes
        print("🔍 Визначення розмірів словників...")
        self.input_dim = self._get_vocab_size(f"{self.lang_keys[0]}_ids")
        self.output_dim = self._get_vocab_size(f"{self.lang_keys[1]}_ids")

        print(f"✅ Input vocab size: {self.input_dim}")
        print(f"✅ Output vocab size: {self.output_dim}")

        self.device = self._get_device()
        print(f"🖥️ Using device: {self.device}")

    def _get_device(self) -> torch.device:
        """Визначає найкращий доступний пристрій"""
        if torch.cuda.is_available():
            return torch.device("cuda")
        elif torch.backends.mps.is_available():
            return torch.device("mps")
        else:
            return torch.device("cpu")

    def _get_vocab_size(self, ids_key):
        """Безпечне визначення розміру словника"""
        sample_size = min(10000, len(self.dataset.dataset))
        max_id = 0

        for i in range(sample_size):
            ids = self.dataset.dataset[i][ids_key]
            if isinstance(ids, list) and len(ids) > 0:
                current_max = max(ids)
                max_id = max(max_id, current_max)

        # +1 бо індексація з 0
        return max_id + 1

    def _cleanup_gpu(self):
        """Очищення пам'яті GPU/MPS"""
        gc.collect()
        if self.device.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        elif self.device.type == "mps":
            torch.mps.empty_cache()

    def objective(self, trial):
        """Оптимізована функція цілі з обробкою помилок"""
        # Повна очистка перед кожним trial
        self._cleanup_gpu()

        # Менші розміри для стабільності
        params = {
            "clip": trial.suggest_float("clip", 0.5, 2.0),
            "n_epochs": 1,  # Тільки 1 епоха для швидкого тестування
            "input_dim": self.input_dim,
            "output_dim": self.output_dim,
            "encoder_embedding_dim": trial.suggest_categorical("encoder_embedding_dim", [64, 128]),
            "decoder_embedding_dim": trial.suggest_categorical("decoder_embedding_dim", [64, 128]),
            "encoder_hidden_dim": trial.suggest_categorical("encoder_hidden_dim", [64, 128]),
            "decoder_hidden_dim": trial.suggest_categorical("decoder_hidden_dim", [64, 128]),
            "model_dir": "./saved_models",
            "pad_index": 1,
            "teacher_forcing_ratio": trial.suggest_categorical("teacher_forcing_ratio", [0.5, 0.75]),
        }

        trainer = None
        try:
            trainer = Seq2SeqTrainer(self.dataset, **params)
            best_loss = trainer.train()

            # Очищення
            del trainer
            self._cleanup_gpu()

            return best_loss

        except RuntimeError as e:
            error_msg = str(e).lower()

            # Обробка CUDA/cuDNN помилок
            if any(keyword in error_msg for keyword in ["cuda", "cudnn", "out of memory", "device-side assert"]):
                print(f"⚠️ GPU error in trial {trial.number}: {e}")
                print("🧹 Cleaning up...")

                if trainer is not None:
                    del trainer

                self._cleanup_gpu()

                # Повідомляємо Optuna про провал
                return float('inf')
            else:
                # Інші помилки пробрасуємо далі
                raise e

        except Exception as e:
            print(f"❌ Unexpected error in trial {trial.number}: {e}")
            if trainer is not None:
                del trainer
            self._cleanup_gpu()
            raise e

    def optimize(self):
        """Запуск оптимізації з обробкою помилок"""
        # Pruner для ранньої зупинки
        pruner = optuna.pruners.MedianPruner(
            n_startup_trials=1,
            n_warmup_steps=3
        )

        study = optuna.create_study(
            direction="minimize",
            pruner=pruner,
            study_name="seq2seq_optimization"
        )

        print(f"🚀 Starting optimization with {self.n_trials} trials...")

        try:
            study.optimize(
                self.objective,
                n_trials=self.n_trials,
                catch=(RuntimeError,),  # Ловимо RuntimeError
                show_progress_bar=True
            )
        except KeyboardInterrupt:
            print("⚠️ Optimization interrupted by user")
        except Exception as e:
            print(f"❌ Optimization failed: {e}")

        # Перевірка результатів
        completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

        if len(completed_trials) == 0:
            print("❌ No trials completed successfully")
            return None

        print(f"\n✅ Completed {len(completed_trials)} out of {len(study.trials)} trials")
        print(f"🏆 Best trial: {study.best_trial.number}")
        print(f"📊 Best loss: {study.best_value:.4f}")

        return study.best_params

In [ ]:
# Токенізувати дані, створити словники та зібрати тренувальний DataLoader.

batch_size = 32

with Tokenizer(
    split_data,
    langs=["en", "pl"],
    space_models=["en_core_web_sm", "pl_core_news_sm"]
) as tokenizer_obj:
    tokenized_data = tokenizer_obj()
    pad_index = tokenizer_obj.pad_index
    langs = tokenizer_obj.langs

with DataLoaderGetter(tokenized_data, pad_index, langs, batch_size=batch_size, shuffle=True) as dataloader:
    train_data_loader = dataloader

print(train_data_loader.dataset[0])

In [ ]:
hp_optimizer = HyperParametersOptimizer(train_data_loader, n_trials=3)
best_params = hp_optimizer.optimize()

print(best_params)